# Affine initialization and B-spline SVF registration

Register the real `r64` acquisition to `r16`, the pair used in the [ANTsX tutorial](https://gist.github.com/ntustison/12a656a5fc2f6f9c4494c88dc09c5621).
ANTsPy downloads the images on first use. First fit an affine, then optimize a
B-spline stationary velocity field. Images are resized to 96 × 96 to shorten
the demonstration while retaining their physical domains.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
from antstorch.bspline_flows import ImageDomain, affine_registration, bspline_svf_registration

def tensor_and_domain(image):
    tensor = torch.from_numpy(image.numpy().T.copy())[None, None].to(device)
    domain = ImageDomain(image.shape, spacing=image.spacing, origin=image.origin,
                         direction=tuple(map(tuple, image.direction)))
    return tensor, domain

fixed_image = ants.image_read(ants.get_ants_data("r16")).clone("float")
moving_image = ants.image_read(ants.get_ants_data("r64")).clone("float")
fixed_image = ants.iMath(ants.resample_image(fixed_image, (96, 96), use_voxels=True), "Normalize")
moving_image = ants.iMath(ants.resample_image(moving_image, (96, 96), use_voxels=True), "Normalize")
fixed, fixed_domain = tensor_and_domain(fixed_image)
moving, moving_domain = tensor_and_domain(moving_image)
affine = affine_registration(
    fixed, moving, fixed_domain, moving_domain, transform_type="Affine", similarity="lncc",
    iterations=(40, 40), learning_rate=0.03, shrink_factors=(2, 1),
    smoothing_sigmas=(1., 0.), multi_start=False, padding_mode="border",
)
result = bspline_svf_registration(
    fixed, moving, fixed_domain, moving_domain,
    initial_affine=(affine["matrix"], affine["translation"]),
    mesh_size=5, iterations=(30, 20), shrink_factors=(2, 1), smoothing_sigmas=(1., 0.),
    optimizer="adam", learning_rate=0.03, similarity="lncc", bending_weight=1e-3,
    squaring_steps=5, padding_mode="border",
)

## Check alignment and local volume changes
Tensor axes are `(batch, channels, y, x)`; displacement components are physical
x and y. `fwdtransforms` contains only the residual SVF displacement, while
the affine is returned separately. The Jacobian refers to the total map.
Do not apply the residual field alone and expect the complete registration.

In [ ]:
warped = result["warpedmovout"]
jacobian = result["jacobian_determinant"]
displacement = result["fwdtransforms"]
initial_mse = (fixed-moving).square().mean().item()
affine_mse = (fixed-affine["warpedmovout"]).square().mean().item()
final_mse = (fixed-warped).square().mean().item()
assert warped.shape == fixed.shape
assert torch.isfinite(warped).all() and torch.isfinite(jacobian).all()
assert displacement.shape == (1, 2, *fixed.shape[2:])
print(f"MSE: unregistered={initial_mse:.6f}; affine={affine_mse:.6f}; affine+SVF={final_mse:.6f}")
print("Nonpositive Jacobian voxels:", int((jacobian <= 0).sum()))
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
for ax, image, title in zip(axes, [fixed, moving, affine["warpedmovout"], warped],
                            ["Fixed", "Moving", "Affine", "Affine + B-spline SVF"]):
    ax.imshow(image[0, 0].cpu(), origin="lower", cmap="gray")
    ax.set_title(title); ax.axis("off")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
stride = 3
gy, gx = np.mgrid[0:fixed.shape[2]:stride, 0:fixed.shape[3]:stride]
field = displacement[0].cpu().numpy()
axes[0].quiver(gx, gy, field[0, ::stride, ::stride], field[1, ::stride, ::stride],
               angles="xy", scale_units="xy", scale=1)
axes[0].set_title("Residual SVF displacement"); axes[0].set_aspect("equal")
view = axes[1].imshow(jacobian[0].cpu(), origin="lower", cmap="coolwarm")
axes[1].set_title("Total-map Jacobian")
fig.colorbar(view, ax=axes[1])
plt.show()

Optimization uses local normalized cross-correlation; the printed MSE is a
separate descriptive measure and need not decrease monotonically. Inspect
anatomical alignment, residual displacements, inverse consistency and nonpositive
Jacobian counts. The short schedule is illustrative, not a claim of optimal
registration or guaranteed absence of folding.